# Manage a quantum-centric supercomputing workflow with qcsc-prefect

*Usage estimate: less than one minute of QPU time on an IBM Quantum Heron processor
(100,000 shots of one 32-qubit circuit). Queue time and classical processing are not
included, and the actual time may vary.*

A quantum-centric supercomputing (QCSC) workflow runs a QPU and an HPC system as one
machine. Such a workflow is hard to run by hand: it spans several schedulers and services,
it has many parameters, and a failure in a late stage can waste the results of the earlier
ones. qcsc-prefect runs QCSC workflows on [Prefect](https://www.prefect.io/), and keeps them
organized, configurable and resumable.

This tutorial shows these features on sample-based quantum diagonalization (SQD) of the N₂
molecule. QRMI samples the circuit inside a Slurm job, and the
[fulqrum](https://github.com/qiskit-community/fulqrum) eigensolver diagonalizes the sampled
subspaces as Slurm jobs, all from one Prefect Cloud deployment.

## Learning outcomes

After completing this tutorial, you will be able to:

1. Organize a QCSC workflow as Prefect tasks, one per Qiskit Patterns stage, and follow it
   in Prefect Cloud.
2. Separate the parameters of a workflow by role: run parameters in the Prefect Cloud run
   form, and the HPC and QPU settings in Prefect blocks.
3. Resume a failed run without repeating its QPU job, and reuse the samples of one run in a
   new run.
4. Move the same workflow to another HPC system by changing blocks only.

## Prerequisites

> **Note**
>
> Complete the [qcsc-prefect setup guide](TODO-qcsc-prefect-setup-guide) first. **(TODO: link to the
> qcsc-prefect setup guide.)** This tutorial assumes that it is done, including:
>
> - a Python environment with qcsc-prefect on the Slurm login node, which serves as the
>   workflow host
> - Prefect Cloud login on the workflow host
> - QRMI with the SPANK plugin on the cluster, and your IBM Quantum credentials for QRMI
> - on the compute nodes, a Python environment for the QRMI worker with QRMI, its
>   `task_runner` and `qcsc-prefect-qiskit`

We also suggest that you are familiar with the following:

- [Sample-based quantum diagonalization of a chemistry Hamiltonian](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization), which
  explains the algorithm that this tutorial runs
- [Run quantum workloads with QRMI](https://quantum.cloud.ibm.com/docs/en/tutorials/run-quantum-workloads-with-qrmi)
- [Prefect concepts](https://docs.prefect.io/v3/concepts): flows, tasks, blocks and
  deployments

## Background

### The challenge of QCSC workflows

The SQD workflow in this tutorial is typical of QCSC workloads:

- **It spans several resources.** A Slurm job holds the QPU and samples a circuit. Then
  several Slurm jobs diagonalize subspaces on the compute nodes, round after round.
- **It has many parameters, at different levels.** Some describe the science (the molecule,
  the subspace dimension), some describe the circuit and its compilation, and some describe
  the cluster (partition, walltime, the path of the solver executable). Credentials must
  stay out of all of them.
- **It is long and can fail anywhere.** A solver job can hit its walltime, a node can fail,
  or the QPU queue can be long. Without care, a failure in the last stage means sampling on
  the QPU again.

qcsc-prefect addresses each of these points.

### An organized workflow

A Prefect *flow* is a Python function made of *tasks*. qcsc-prefect makes each stage of a
[Qiskit pattern](https://quantum.cloud.ibm.com/docs/en/guides/intro-to-patterns), **Map**,
**Optimize**, **Execute** and **Post-process**, one task. Prefect Cloud then shows, for every
run:

- the task graph, with an arrow wherever a task passes its result to another task
- each task's state, inputs, logs and duration, filterable by the stage tag
- *artifacts*: tables and links that tasks publish, such as where the samples came from or
  the energy of each subspace

The workflow host only submits and watches work. The heavy lifting runs as Slurm jobs:

```text
┌─ Where the work runs ───────────────────────────────────────────────────┐
│                                                                         │
│  ┌─────────────────────────┐                                            │
│  │ Prefect Cloud           │                                            │
│  │ runs, logs, artifacts   │                                            │
│  └────────────┬────────────┘                                            │
│               │ the host polls for runs and reports states and logs     │
│  ┌────────────┴────────────┐                                            │
│  │ Workflow host           │                                            │
│  │ serves the deployment   │                                            │
│  └──┬──────────────────────┘                                            │
│     │                                                                   │
│     ├── sbatch --qpu ──►┌────────────────────┐     ┌─────────────────┐  │
│     │                   │ Slurm job:         │◄───►│ IBM Quantum QPU │  │
│     │                   │ QRMI worker,       │     └─────────────────┘  │
│     │                   │ compile + sample   │                          │
│     │                   └────────────────────┘                          │
│     │                                                                   │
│     └── sbatch ────────►┌────────────────────┐                          │
│                         │ Slurm jobs:        │                          │
│                         │ fulqrum, one per   │                          │
│                         │ subspace           │                          │
│                         └────────────────────┘                          │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Parameters by role

qcsc-prefect keeps each kind of setting where it belongs:

| Layer | Examples | Changes | Where it lives |
|---|---|---|---|
| Run parameters | molecule, shots, subspace dimension | every run | the Prefect Cloud run form |
| Blocks | partition, walltime, threads, executable paths, QPU resource | per cluster | Prefect blocks, created once |
| Credentials | IBM Quantum API key and CRN | rarely | QRMI and SPANK on the cluster, set up by the setup guide |

The run form groups the run parameters by stage, so a run is described by what you study
rather than by how the cluster works. The blocks describe each job in three layers:

- a **CommandBlock** says *what* runs: the executable and its default arguments
- an **ExecutionProfileBlock** says *how* it runs: nodes, threads, walltime, launcher,
  modules and environment
- an **HPCProfileBlock** says *where* it runs: the scheduler, partition, account and the
  path of each executable on that system

The flow loads blocks by name when it runs. To move the workflow to another system, such as
Miyabi (PBS) or Fugaku (PJM), you create blocks for that system, and the flow and its run
parameters do not change.

### Resumable execution

The tasks of the first three stages *persist* their results, so Prefect stores them with the
flow run. If a later task fails, **Retry** on the flow run reuses the stored results and runs
only what failed:

```text
┌─ Retry of a failed run ──────────────────────────────────────────┐
│                                                                  │
│  Run                                                             │
│    Map ───► Optimize ───► Execute ───────► Post-process          │
│    done     done          done, QPU job    FAILED (walltime)     │
│                                                                  │
│  Retry of the same run, after raising the walltime in the block  │
│    Map ───► Optimize ───► Execute ───────► Post-process          │
│    stored   stored        stored,          runs again, done      │
│                           no QPU job                             │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

A *new* run can also start from earlier samples: the Execute stage records where its samples
are, and a run that points to them loads them instead of sampling.

### SQD at a glance

SQD samples bitstrings from a LUCJ circuit built from CCSD amplitudes, fixes the
bitstrings that noise has corrupted (*configuration recovery*), and diagonalizes the
Hamiltonian in subspaces spanned by the sampled configurations. The
[SQD tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization) explains the algorithm. In qcsc-prefect, it is the
flow `sqd`, with this task graph:

```text
┌─ Task graph of sqd ──────────────────────────────────────────────────────┐
│                                                                          │
│  ┌─ Map ───────────────────────┐                                         │
│  │ compute_molecular_integrals ├───────────────┐                         │
│  └──────────────┬──────────────┘               │                         │
│                 │ integrals                    │                         │
│                 ▼                              │                         │
│  ┌─ Optimize ──────────────────┐               │                         │
│  │ get_lucj_params             │               │                         │
│  └──────────────┬──────────────┘               │                         │
│                 │ LUCJ parameters              │ integrals               │
│                 ▼                              │                         │
│  ┌─ Execute ───────────────────┐               │                         │
│  │ quantum_sampling            │               │                         │
│  │  └ build_lucj_circuit       │               │                         │
│  └──────────────┬──────────────┘               │                         │
│                 │ samples                      │                         │
│                 ▼                              │                         │
│  ┌─ Post-process ──────────────┐               │                         │
│  │ sqd_loop                    │◄──────────────┘                         │
│  └─────────────────────────────┘                                         │
│                                                                          │
│  Reusing samples: load_qrmi_result or load_runtime_job replaces          │
│  get_lucj_params and quantum_sampling.                                   │
│  Random mode: no Optimize or Execute task; Map feeds sqd_loop directly.  │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

The example is N₂ at a bond length of 1.1 Å in the 6-31G basis, with the two 1s core orbitals
frozen: 16 spatial orbitals with 5 α and 5 β electrons, which is a 32-qubit circuit.

### What comes next

- **Setup** installs the SQD workflow, creates its blocks and deploys it.
- **Part 1** goes through the four stages one at a time. Each step computes the stage in this
  notebook, then shows how qcsc-prefect organizes it, which parameters control it, and how
  it resumes.
- **Part 2** runs the whole workflow from Prefect Cloud in one click and reads its results.
- **An optional section** retries a failed run, reuses its samples, and moves the workflow
  to another HPC system.

## Setup: the SQD workflow and its Prefect blocks

Run these steps once, from the root of the qcsc-prefect checkout on the workflow host. The
checkout must be on a file system that the compute nodes see at the same path.

### Step 1. Install the SQD workflow

On the **workflow host**, install the workflow with its `fulqrum` and `qrmi` extras:

In [ ]:
%%bash
# Run in a terminal on the workflow host, not in this notebook.
cd /path/to/qcsc-prefect

uv pip install \
  -e 'packages/qcsc-prefect-qiskit[slurm]' \
  -e packages/qcsc-prefect-fulqrum \
  -e algorithms/qcsc_workflow_utility \
  -e algorithms/open_loop/sqd

On the **compute nodes**, the QRMI worker environment from the setup guide also needs
the SQD compilation hook, from the same revision as the workflow host. The hook imports
only Qiskit, so ffsim and fulqrum are not needed there. With that environment active,
install the hook and check the imports. The path of this Python is `python_executable`
in Step 3:

In [ ]:
%%bash
# Run in a terminal, with the compute-node Python environment active.
uv pip install --no-deps -e algorithms/open_loop/sqd
python -c "import qrmi, qcsc_prefect.integrations.qiskit.qrmi.worker, sqd_common.transpile"
which python task_runner

Check the installation on the workflow host:

In [ ]:
%%bash
# Run in a terminal on the workflow host, not in this notebook.
uv pip list | grep -E "qcsc|sqd"
which sqd-deploy sqd-create-blocks

### Step 2. Build fulqrum for the compute nodes

fulqrum is built per site into its own virtual environment. It needs a C++ compiler with
OpenMP, and no Prefect. `FULQRUM_WITH_PRIMME=1` also builds the PRIMME eigensolver, which
needs a C compiler, Python headers, `pkg-config` and a BLAS/LAPACK with a `.pc` file:

In [ ]:
%%bash
# Run in a terminal on a node that can build for the compute nodes.
FULQRUM_WITH_PRIMME=1 packages/qcsc-prefect-fulqrum/native/build_fulqrum.sh /opt/venvs/fulqrum
/opt/venvs/fulqrum/bin/python -m pip install --no-deps packages/qcsc-prefect-fulqrum
realpath /opt/venvs/fulqrum/bin/qcsc-fulqrum-solve

Record the path of `qcsc-fulqrum-solve`, because the solver blocks point to it. See
[Fulqrum integration](https://qiskit-community.github.io/qcsc-prefect/howto/howto_use_fulqrum_prefect/) for the build options.

### Step 3. Configure and create the Prefect blocks

Create a shared working directory and copy the example configuration:

In [ ]:
%%bash
# Run in a terminal on the workflow host, not in this notebook.
mkdir -p /shared/sqd-fulqrum
cp algorithms/open_loop/sqd/sqd_blocks.example.toml \
   algorithms/open_loop/sqd/sqd_blocks.toml

`sqd_blocks.toml` keeps the settings you edit at the top. Edit them:

| Parameter | Example | Description |
|---|---|---|
| `solver` | `"fulqrum"` | Subspace diagonalizer: `fulqrum` or `dice` |
| `profile` | `"slurm"` | System profile: `slurm`, `miyabi`, `fugaku`, or a path to your own |
| `queue` | `"normal"` | Slurm partition for the solver jobs |
| `project` | `"my-account"` | Optional Slurm account (`#SBATCH --account`) |
| `work_dir` | `"/shared/sqd-fulqrum"` | Job directories, shared by the host and the nodes |
| `fulqrum_executable` | `"/opt/venvs/fulqrum/bin/qcsc-fulqrum-solve"` | The path from Step 2 |
| `eigensolver` | `"primme"` | fulqrum's sparse solver: `primme` (built in Step 2) or `scipy` |
| `[qrmi] resource` | `"ibm_kingston"` | QPU resource, the Slurm `--qpu` value |
| `[qrmi] python_executable` | `"/opt/venvs/qrmi/bin/python"` | The compute-node Python checked in Step 1 |

The `[qrmi]` rows are keys of the `[qrmi]` table further down the file, which is active by
default. Replace its `resource` placeholder with your QPU, and keep any other top-level
setting above the first table header. The QPU credentials stay in the cluster's QRMI
configuration, so the file holds no secret.

The **profile** holds the scheduler and resource settings of your system, such as the HPC
target, nodes, threads, walltime and modules, and the walltime and threads of the QRMI job.
The `slurm` profile suits a generic Slurm cluster, and any key you set in your file
overrides it. Rarely changed settings, such as block names, timeouts and solver tuning,
are in a commented `[advanced]` table at the end of the file. See the
[cluster profile guide](TODO-qcsc-prefect-cluster-profile-guide) **(TODO: link)** to write
a profile for another system.

Then create the blocks:

In [ ]:
%%bash
# Run in a terminal on the workflow host, not in this notebook.
sqd-create-blocks --config algorithms/open_loop/sqd/sqd_blocks.toml

The command saves two sets of the three block layers, one for the solver jobs and one for
the QRMI sampler job, which requests the QPU, and a `qrmi-runtime` block that names the
QRMI set. The `sqd-fulqrum-solver` block, a
`FulqrumSolverJob`, wraps the solver set and holds the eigensolver settings (here PRIMME).
The flow calls it once per subspace. `prefect block ls` and the **Blocks** page of Prefect Cloud list them:

```text
Type                 Name                          Slug
HPC Command          cmd-sqd-fulqrum-solver        hpc-command/cmd-sqd-fulqrum-solver
Execution Profile    exec-sqd-fulqrum-slurm        execution-profile/exec-sqd-fulqrum-slurm
HPC Profile          hpc-slurm-sqd-fulqrum         hpc-profile/hpc-slurm-sqd-fulqrum
Fulqrum Solver Job   sqd-fulqrum-solver            fulqrum-solver-job/sqd-fulqrum-solver
HPC Command          cmd-sqd-fulqrum-qrmi          hpc-command/cmd-sqd-fulqrum-qrmi
Execution Profile    exec-sqd-fulqrum-qrmi         execution-profile/exec-sqd-fulqrum-qrmi
HPC Profile          hpc-slurm-sqd-fulqrum-qrmi    hpc-profile/hpc-slurm-sqd-fulqrum-qrmi
QRMI Runtime         qrmi-runtime                  qrmi-runtime/qrmi-runtime
```

> **Image placeholder** `docs/images/img-qcsc-prefect-blocks.png`: the blocks of this workflow in Prefect Cloud: the block list, and the three layers of the solver job (what, how, where).
>
> *Capture proposal:* one composite figure. Left: the **Blocks** page, filtered to `sqd-fulqrum`, listing the solver and QRMI blocks, and `qrmi-runtime`. Right: the `sqd-fulqrum-solver` block page, which shows its eigensolver settings and links the command, execution profile and HPC profile blocks in one view. If the solver block page does not show the linked blocks' fields, use the `exec-sqd-fulqrum-slurm` page instead, because it holds the walltime that the optional Retry section changes.

The `qrmi-runtime` block (a `QrmiRuntimeBlock` with `scheduler = "slurm"`) is the
default **QRMI Runtime Block** of the run form, so a run samples through these QRMI
blocks without any QRMI setting in the form.

### Step 4. Deploy the workflow

`sqd-deploy` serves the flow from the workflow host. The serving process picks up
runs started from Prefect Cloud and must keep running, for example under `nohup`, `tmux` or
`screen`:

In [ ]:
%%bash
# Run in a terminal on the workflow host, not in this notebook.
nohup sqd-deploy > sqd-deploy.log 2>&1 &
tail -f sqd-deploy.log

When the log says that the flow `sqd` is being served, the deployment
appears in Prefect Cloud under **Deployments**.

## Part 1: the pattern step by step

Part 1 computes the four stages in this notebook with PySCF, ffsim, Qiskit, QRMI and
qiskit-addon-sqd, without Prefect. After each stage, a box describes how qcsc-prefect runs
that stage in the workflow:

- **Organize**: the task, its place in the graph, and what it shows in Prefect Cloud
- **Parameters**: which part of the run form, or which block, controls the stage
- **Resume**: what Prefect stores, and what happens on Retry

In the workflow, every run parameter is a field of the deployment's **Custom run** form,
grouped by stage. A small screenshot after each box shows one qcsc-prefect feature at work
in that stage.

The code cells need the packages below in the notebook's kernel, and the QRMI credentials
from the setup guide. See the [SQD tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization) for the details of each
computation.

In [ ]:
%%bash
# Run once in a terminal, in the Python environment of this notebook's kernel.
python -m pip install "qrmi[ibm]" pyscf ffsim qiskit-addon-sqd python-dotenv matplotlib

In [ ]:
import math
import os
from functools import partial

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
from dotenv import load_dotenv
from pyscf import ao2mo, cc, mcscf, scf
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian, solve_sci_batch
from qrmi import QRMIService
from qrmi.primitives.ibm import SamplerV2, get_backend

# QRMI settings from the setup guide. Variables already set in the environment win.
load_dotenv(override=False)
os.environ.setdefault("QRMI_JOB_QPU_RESOURCES", "ibm_kingston")
os.environ.setdefault("QRMI_JOB_QPU_TYPES", "ibm-quantum-compute-service")
print(f"QPU resource: {os.environ['QRMI_JOB_QPU_RESOURCES']}")

### Step 1. Map: `compute_molecular_integrals`

```text
┌─ You are here: Map ───────────────────────────────────────┐
│                                                           │
│  ╔═════╗   ┌──────────┐   ┌─────────┐   ┌──────────────┐  │
│  ║ Map ╟──►│ Optimize ├──►│ Execute ├──►│ Post-process │  │
│  ╚══╤══╝   └──────────┘   └─────────┘   └───────▲──────┘  │
│     └───────────────────────────────────────────┘         │
│                                                           │
└───────────────────────────────────────────────────────────┘
```

The Map stage turns the molecule into the inputs of all later stages. RHF gives the
molecular orbitals. The active-space integrals leave out the two frozen 1s cores. CCSD gives
the $t_2$ amplitudes of the ansatz, and its density matrix gives the orbital occupancies
that start configuration recovery. CCSD is restarted `ccsd_maxiter` times from its own
amplitudes, so the result does not depend on the history of the first run's DIIS
extrapolation.

In [ ]:
ATOM = "N 0 0 0; N 0 0 1.1"
BASIS = "6-31g"
N_FROZEN = 2
CCSD_MAXITER = 4

mol = pyscf.gto.M(atom=ATOM, basis=BASIS, symmetry=False, verbose=0)
mf = scf.RHF(mol).run()
norb = mol.nao_nr() - N_FROZEN
nelec = (mol.nelec[0] - N_FROZEN, mol.nelec[1] - N_FROZEN)

# Active-space Hamiltonian with the 1s cores frozen
cas = mcscf.CASCI(mf, ncas=norb, nelecas=nelec)
one_body_tensor, core_energy = cas.get_h1eff()
two_body_tensor = ao2mo.restore(1, cas.get_h2eff(), norb)

# CCSD, restarted from the previous amplitudes
ccsd = cc.CCSD(mf, frozen=N_FROZEN)
eris = ccsd.ao2mo()
ccsd.kernel(eris=eris)
for _ in range(CCSD_MAXITER - 1):
    ccsd.kernel(ccsd.t1, ccsd.t2, eris=eris)
t2 = ccsd.t2

occupancies = np.diag(ccsd.make_rdm1())[N_FROZEN:] / 2
initial_occupancies = (occupancies, occupancies)

print(f"norb = {norb}, nelec = {nelec}")
print(f"E(RHF)  = {mf.e_tot:.8f} Ha")
print(f"E(CCSD) = {ccsd.e_tot:.8f} Ha")

> **How qcsc-prefect manages this step**
>
> **Organize.** The task `compute_molecular_integrals` has the tag `map`. Its result goes to
> two tasks, `get_lucj_params` and `sqd_loop`, so the graph shows two arrows leaving it. The
> PySCF output appears in the task's logs.
>
> **Parameters.** The **Molecule** group of the run form selects the problem: an FCIDUMP
> file, or a geometry and basis, plus the number of CCSD restarts. The workflow reads this
> Hamiltonian from `algorithms/open_loop/sqd/data/n2_631g_fc.fcidump`, which has the cores frozen
> already.
>
> **Resume.** The result is persisted (`compressed/json`). A Retry of the run loads it
> instead of running PySCF again.

> **Image placeholder** `docs/images/img-qcsc-prefect-step-map.png`: **Organize**: the integrals feed two stages.
>
> *Capture proposal:* a crop of the flow run graph around `compute_molecular_integrals`, with its two outgoing arrows to `get_lucj_params` and `sqd_loop`, and the task selected so that the side panel shows its `map` tag and Completed state. This shows that the graph follows the data, not just the call order.

### Step 2. Optimize: `get_lucj_params`

```text
┌─ You are here: Optimize ──────────────────────────────────┐
│                                                           │
│  ┌─────┐   ╔══════════╗   ┌─────────┐   ┌──────────────┐  │
│  │ Map ├──►║ Optimize ╟──►│ Execute ├──►│ Post-process │  │
│  └──┬──┘   ╚══════════╝   └─────────┘   └───────▲──────┘  │
│     └───────────────────────────────────────────┘         │
│                                                           │
└───────────────────────────────────────────────────────────┘
```

The Optimize stage fits the ansatz to the QPU before any circuit exists. ffsim builds the
LUCJ operator from $t_2$, keeping only interactions that a heavy-hex lattice runs without
SWAP gates: same-spin pairs between neighboring orbitals, and opposite-spin pairs on every
fourth orbital, at most five of them on a heavy-hex (Heron) processor. ffsim then
*compresses* the operator (`optimize=True`, on by default in the workflow) so that one layer
reproduces $t_2$ more closely.

In [ ]:
N_REPS = 1

pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p) for p in range(0, norb, 4)][:5]

lucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2,
    n_reps=N_REPS,
    interaction_pairs=(pairs_aa, pairs_ab),
    optimize=True,
    options={"maxiter": 20},
)
n_params = ffsim.UCJOpSpinBalanced.n_params(norb, N_REPS, interaction_pairs=(pairs_aa, pairs_ab))

print(f"Opposite-spin pairs: {pairs_ab}")
print(f"Ansatz parameters: {n_params}")

> **How qcsc-prefect manages this step**
>
> **Organize.** The task `get_lucj_params` has the tag `optimize`. Its result carries
> everything the circuit needs, so one arrow leads from it to `quantum_sampling`.
>
> **Parameters.** The **Circuit** group of the run form sets the number of LUCJ layers,
> compression (on by default) and its iteration limit (20). The same group has a
> **Configure Transpiler** switch, off by default: switch it on to change the optimization
> level and the SABRE layout search that the Execute stage uses.
>
> **Resume.** The result is persisted (`compressed/pickle`). A long compression runs once
> per run, even if the run is retried.

> **Image placeholder** `docs/images/img-qcsc-prefect-step-optimize.png`: **Parameters**: the Circuit group of the run form.
>
> *Capture proposal:* a crop of the custom-run form with the **Circuit** group expanded: LUCJ layers, compression and its iteration limit, reset mitigation, and **Configure Transpiler** switched on so that the transpiler settings show, each with its description. This shows that circuit knobs are typed, documented fields grouped by stage, not script edits.

### Step 3. Execute: `quantum_sampling`

```text
┌─ You are here: Execute ───────────────────────────────────┐
│                                                           │
│  ┌─────┐   ┌──────────┐   ╔═════════╗   ┌──────────────┐  │
│  │ Map ├──►│ Optimize ├──►║ Execute ╟──►│ Post-process │  │
│  └──┬──┘   └──────────┘   ╚═════════╝   └───────▲──────┘  │
│     └───────────────────────────────────────────┘         │
│                                                           │
└───────────────────────────────────────────────────────────┘
```

The Execute stage builds the circuit (Hartree–Fock state, LUCJ operator, measurement),
compiles it for the QPU, and samples it. ffsim's `PRE_INIT` passes run first in the
transpiler and turn the ffsim gates into efficient native-gate sequences.

In [ ]:
qrmi = QRMIService().resources()[0]
backend = get_backend(qrmi)

qreg = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qreg, ClassicalRegister(2 * norb, name="meas"))
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qreg)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(lucj_op), qreg)
circuit.measure(qreg, circuit.cregs[0])

pass_manager = generate_preset_pass_manager(
    optimization_level=3, target=backend.target, seed_transpiler=24
)
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit = pass_manager.run(circuit)
print(f"Backend: {backend.name}, ISA circuit depth: {isa_circuit.depth()}")

In [ ]:
SHOTS = 100_000

sampler = SamplerV2(qrmi, options={"default_shots": SHOTS})
job = sampler.run([(isa_circuit,)])
bool_array = job.result()[0].data.meas.to_bool_array()

# The right half of each bitstring is alpha, the left half is beta.
n_alpha, n_beta = nelec
valid = (bool_array[:, norb:].sum(axis=1) == n_alpha) & (bool_array[:, :norb].sum(axis=1) == n_beta)
print(f"QRMI job {job.job_id()}: {len(bool_array)} shots")
print(f"Correct electron numbers: {valid.mean():.4f}")

> **Note**
>
> To continue without a QPU, replace the sampling cell with uniform random bitstrings. The
> workflow does the same with **Quantum Source** set to `random`:
>
> ```python
> from qiskit_addon_sqd.counts import generate_bit_array_uniform
>
> bool_array = generate_bit_array_uniform(SHOTS, 2 * norb, rand_seed=24).to_bool_array()
> ```

> **How qcsc-prefect manages this step**
>
> **Organize.** The task `quantum_sampling` has the tag `execute`, and the task
> `build_lucj_circuit` runs inside it. With QRMI, the workflow host prepares the circuit
> and submits a Slurm job that requests the QPU. On the compute
> node, the SPANK plugin opens the QPU session, and the QRMI worker compiles the circuit
> against the QPU's current Target, with the same pass manager and seed, and samples it. The
> task logs show the compiled circuit, and the `qrmi-slurm-job` artifact shows the Slurm job.
>
> **Parameters.** The **Sampling** group of the run form sets the shots and the source of the
> samples: the QRMI runtime block saved in Setup Step 3 (`qrmi-runtime`), the job of an
> earlier run, or random bitstrings. The QRMI blocks hold where and how the sampler job runs, and the
> credentials stay on the cluster.
>
> **Resume.** The samples are persisted (`compressed/pickle`), so a Retry submits no new QPU
> job. The `sqd-sampling-source` artifact records where the samples are, so a new run can
> load them, with the task `load_qrmi_result` or `load_runtime_job` in place of
> `quantum_sampling`.

> **Image placeholder** `docs/images/img-qcsc-prefect-step-execute.png`: **Organize** across the QPU boundary: the QPU job, inside the workflow.
>
> *Capture proposal:* a crop of the task-run page of `quantum_sampling`: its logs, with the lines that report the Slurm job and the depth and gate counts of the circuit compiled on the compute node. If the nested `build_lucj_circuit` task shows in the same view, keep it.

### Step 4. Post-process: `sqd_loop`

```text
┌─ You are here: Post-process ──────────────────────────────┐
│                                                           │
│  ┌─────┐   ┌──────────┐   ┌─────────┐   ╔══════════════╗  │
│  │ Map ├──►│ Optimize ├──►│ Execute ├──►║ Post-process ║  │
│  └──┬──┘   └──────────┘   └─────────┘   ╚═══════▲══════╝  │
│     └───────────────────────────────────────────┘         │
│                                                           │
└───────────────────────────────────────────────────────────┘
```

The Post-process stage runs self-consistent configuration recovery. Each round corrects the
sampled bitstrings with the current orbital occupancies, draws several batches of
configurations, diagonalizes the Hamiltonian in each batch's subspace, and updates the
occupancies from the best result. Here, `diagonalize_fermionic_hamiltonian` solves the
batches in this process, with a subspace dimension small enough for a laptop.

In [ ]:
SUBSPACE_DIM = 100_000
NUM_BATCHES = 3
MAX_ITERATIONS = 5

round_energies = []


def callback(results):
    round_energies.append(min(result.energy for result in results) + core_energy)
    print(f"Round {len(round_energies)}: {round_energies[-1]:.8f} Ha")


result = diagonalize_fermionic_hamiltonian(
    one_body_tensor,
    two_body_tensor,
    bool_array,
    samples_per_batch=1000,
    norb=norb,
    nelec=nelec,
    num_batches=NUM_BATCHES,
    max_iterations=MAX_ITERATIONS,
    sci_solver=partial(solve_sci_batch, spin_sq=0.0),
    symmetrize_spin=True,
    max_dim=math.isqrt(SUBSPACE_DIM),
    initial_occupancies=initial_occupancies,
    callback=callback,
    seed=np.random.default_rng(24),
)
print(f"SQD energy: {result.energy + core_energy:.8f} Ha")

The plot compares each round with the full-CI energy of this active space,
−109.10336546 Ha, which PySCF computes in about a minute (`cas.run()` on the `CASCI` object
above). The dashed line is chemical accuracy, 1.6 mHa.

In [ ]:
E_FCI = -109.10336546

rounds = range(1, len(round_energies) + 1)
plt.semilogy(rounds, np.abs(np.array(round_energies) - E_FCI), "o-", label="SQD")
plt.axhline(abs(ccsd.e_tot - E_FCI), color="tab:gray", linestyle=":", label="CCSD")
plt.axhline(1.6e-3, color="tab:red", linestyle="--", label="chemical accuracy")
plt.xlabel("Recovery round")
plt.ylabel("|E - E(FCI)| (Ha)")
plt.xticks(list(rounds))
plt.legend()
plt.show()

> **How qcsc-prefect manages this step**
>
> **Organize.** The task `sqd_loop` has the tag `post-process`. It receives the samples and
> the integrals, so the graph shows two arrows into it. Each batch is one `FulqrumSolverJob`
> call, a `solve_eigenstate` task that submits one Slurm job running fulqrum. The batches of a round run concurrently
> on the compute nodes. The `sqd-configuration-recovery` artifact lists the energy
> and dimension of every subspace in every round.
>
> **Parameters.** The **SQD** group of the run form sets the subspace dimension, the number
> of batches and the number of rounds. **Solver Block** (`solver_block_ref`) selects the
> solver: `fulqrum_solver_job/sqd-fulqrum-solver` by default, or a DICE or SBD block. The eigensolver
> (PRIMME), threads and walltime live in the `sqd-fulqrum-solver` block and its execution
> profile, out of the run form. The flow loads the solver block before any other stage, so a
> wrong block name fails the run before it uses the QPU.
>
> **Resume.** `sqd_loop` is not cached, but it saves a checkpoint after every round: the
> occupancies that seed the next round and the best energy so far, in
> `sqd_checkpoints/<flow run id>/ckpt_t00_w0.npz` under the solver's `work_dir`. On Retry it
> runs again from the stored samples and integrals, with the blocks as they are at that
> time, and continues after the last completed round. Each solver job leaves a
> `job_<timestamp>_<id>` directory under `work_dir` with its script, inputs, output and
> Slurm logs.

> **Image placeholder** `docs/images/img-qcsc-prefect-step-postprocess.png`: **Organize** at HPC scale: one task per solver job, running concurrently.
>
> *Capture proposal:* a crop of the flow run graph (timeline view) under `sqd_loop`, where each solver call is a `solve_eigenstate` task. The bars of one round overlap in time, and the rounds follow one another. This shows concurrent Slurm jobs tracked one by one.

### Summary of Part 1

| Stage | Task (tag) | Parameters | Stored for Retry |
|---|---|---|---|
| Map | `compute_molecular_integrals` (`map`) | Molecule group | integrals, amplitudes, occupancies |
| Optimize | `get_lucj_params` (`optimize`) | Circuit group | LUCJ parameters |
| Execute | `quantum_sampling` (`execute`) | Sampling group, QRMI blocks | samples, and their source as an artifact |
| Post-process | `sqd_loop` (`post-process`) | SQD group, solver blocks | a checkpoint after every round (on by default) |

The parameters you change between runs are in the run form, grouped by stage. The cluster
settings are in blocks, and the credentials are on the cluster. The expensive QPU result is
stored as soon as it exists.

## Part 2: run the whole workflow in one click

Part 2 runs all four stages with one action in Prefect Cloud. The blocks from Setup Step 3
already describe the cluster and the QPU. A run needs only the run parameters.

### Submit the run

In Prefect Cloud, open **Deployments → sqd → Run → Custom run**. Set the
fields below, keep the defaults for the others, and click **Submit**. Only the molecule has
no default; the form still marks the three SQD fields as required so that they stand out:

| Group | Field | Value |
|---|---|---|
| Molecule | FCIDump File | `/path/to/qcsc-prefect/algorithms/open_loop/sqd/data/n2_631g_fc.fcidump` |
| Sampling | Quantum Source | `real-device` |
| Sampling | Shots | `10000` (the default) |
| Sampling | QRMI Runtime Block | `qrmi-runtime` (the default, from Setup Step 3) |
| SQD | Subspace Dimension | `90000` (the default) |
| SQD | Batch Number | `4` (the default) |
| SQD | Max Iteration | `5` (the default) |
| SQD | Checkpoint Recovery | on (the default) |
| — | Solver Block | `fulqrum_solver_job/sqd-fulqrum-solver` (the default) |

Use an absolute path for the FCIDUMP file, because the serving process does not run from the
repository root. For scripted runs, the same parameters are in
`algorithms/open_loop/sqd/default_params/n2.json`.

**Checkpoint Recovery** is on by default and costs little disk space. The checkpoint holds
two occupancy vectors and the best energy, about 16 bytes per orbital plus a few kB,
rewritten in place after each round: under 1 MB even for 1000 orbitals. Each run adds one
such folder under `work_dir/sqd_checkpoints/`, and nothing removes it. The solver job
directories next to it take far more space.

> **Image placeholder** `docs/images/img-qcsc-prefect-submit.png`: the custom-run form of `sqd`, filled in with the values above.
>
> *Capture proposal:* one tall screenshot of the form with the groups Molecule, Circuit, Sampling and SQD expanded and the **Submit** button visible. Collapse the long Circuit group if the form does not fit, because the Optimize step already shows it.

> **Note**
>
> For a first run without the QPU, set **Quantum Source** to `random`. The flow uses uniform
> random bitstrings, skips the Optimize and Execute stages, and still runs the solver jobs.
> To sample from the workflow host instead of a Slurm job, save a `QrmiRuntimeBlock` with
> `scheduler = "iqp"` (the `[runtime]` table of the block config) and name it in
> **QRMI Runtime Block**.

### Read the results

Everything about the run is on its flow run page: the task graph and its states, the logs,
and the artifacts.

> **Image placeholder** `docs/images/img-qcsc-prefect-run.png`: the flow run page of a completed run: the task graph with the four stages and the two arrows into `sqd_loop`, with the logs below it.
>
> *Capture proposal:* one full-window screenshot of the flow run page. The graph and the state of every task are at the top. Below them, open the **Logs** tab with the `quantum_sampling` lines that report the circuit compiled on the compute node, or the `sqd_loop` lines with the energy of each round. Expand `quantum_sampling` in the graph, if the UI allows it, so that `build_lucj_circuit` shows. Is this one screen readable at docs width, or should the logs be a second crop?

The run publishes three artifacts, listed in the run's **Artifacts** tab:

| Artifact | Published by | Contents |
|---|---|---|
| `sqd-sampling-source` | Execute | where the samples are (QRMI work directory, or IBM Quantum job ID for IQP), resource, shots |
| `qrmi-slurm-job` | Execute | the Slurm job of the QRMI sampler: ID, state, work directory |
| `sqd-configuration-recovery` | Post-process | energy and dimension of every subspace in every round |

> **Image placeholder** `docs/images/img-qcsc-prefect-artifacts.png`: the artifacts of the run.
>
> *Capture proposal:* one composite figure. Left: the run's **Artifacts** tab listing the three artifacts. Right: the `sqd-configuration-recovery` table opened, with `sqd-sampling-source` below it if both fit. The artifacts have keys, so the top-level **Artifacts** page also lists their latest versions across runs, which could show all three in one screen instead. Which view reads better?

The energy reported by the flow is the lowest energy in the configuration-recovery table.
On the cluster, each solver job leaves a `job_<timestamp>_<id>` directory under `work_dir`,
with the rendered Slurm script, `input.npz`, `solver.json`, the Slurm output, and either
`output.npz` and `result.json`, or `error.json`. Use `sacct` to see the jobs of the run:

In [ ]:
%%bash
# Run in a terminal on the workflow host, not in this notebook.
sacct --format=JobID,JobName,State,ExitCode,Elapsed,WorkDir%60

## Optional: recover, reuse and move the workflow

### Retry a failed run, and reuse its samples

**Retry.** Suppose a solver job hits its walltime in round 4 of 5, and `sqd_loop` fails.
Raise the walltime in the `exec-sqd-fulqrum-slurm` block under **Blocks**, then click
**Retry** on the failed flow run. The retried run loads the stored results of the first
three stages, so it submits no QPU job. It runs only `sqd_loop` again, with the new
walltime, and starts at round 4 from the checkpoint of round 3. Its log shows
`[ckpt] RESUME from …: completed 3 step(s)`, then `SQD iteration 4/5`.

**Continue a recovery with more rounds.** The `sqd-checkpoint` artifact of a run gives the
path of its checkpoint. To continue that recovery in a new run, start a custom run with the
same parameters, but:

- set **Resume from Checkpoint** (`resume_checkpoint`) to that path
- raise **Max Iteration**

The new run starts after the completed rounds. If the checkpoint has 32 of 100 rounds done
and the new run has Max Iteration `200`, it continues at round 33/200, from the stored
occupancies and best energy. The samples, the molecule, **Subspace Dimension**,
**Batch Number** and the solver type (the `fulqrum_solver_job` part of **Solver Block**)
must be the same as in the first run, so reuse its samples as described below. The solver
block's name and job settings, such as its walltime, may change. Otherwise the run fails
before any solver job and names what differs. The new run
only reads the given checkpoint and writes its own.

**Reuse.** A new run computes everything again unless you point it to earlier samples. To
try another subspace dimension with the same samples, start a custom run with the same
parameters, but:

- set **Load from QRMI Job Directory** (`qrmi_result_dir`) to the work directory in the first run's
  `sqd-sampling-source` artifact
- change **Subspace Dimension**

The Execute stage of the new run is the task `load_qrmi_result`, which reads the samples and
submits nothing to the QPU. For samples taken through IQP, set
**Load from Qiskit Runtime Job ID** (`qiskit_runtime_job_id`) instead, and the stage is
`load_runtime_job`.

> **Image placeholder** `docs/images/img-qcsc-prefect-retry-reuse.png`: a retried run, and a run that reuses samples.
>
> *Capture proposal:* one image with two graphs side by side. Left: the graph of a retried run, where the first three tasks completed from stored results and only `sqd_loop` ran again, with its `[ckpt] RESUME` log line. Right: the graph of a reuse run, with `load_qrmi_result` in place of `get_lucj_params` and `quantum_sampling`. Alternatively, the **Runs** list filtered to this deployment shows the original, retried and reuse runs together with their states and durations. Would that single list be clearer than the two graphs?

### Run on another HPC system

The flow loads its blocks by name, so moving it to another system means creating the same
blocks there. On Miyabi (PBS) or Fugaku (PJM), build fulqrum with PRIMME on that system as
in Setup Step 2, and fill in your settings as in Setup Step 3 with `profile = "miyabi"` or
`profile = "fugaku"` and your group as `project`. Then run `sqd-create-blocks` again. For
another system, write your own profile file and give its path as `profile` (see the
[cluster profile guide](TODO-qcsc-prefect-cluster-profile-guide) **(TODO: link)**).

The flow, its deployment and the run form stay the same. Only the HPC Profile and Execution
Profile blocks differ: the scheduler, the queue or resource group, the modules, and the path
of `qcsc-fulqrum-solve` on that system.

> **Image placeholder** `docs/images/img-qcsc-prefect-blocks-other-hpc.png`: the HPC Profile block of the solver jobs, for Slurm and for another system.
>
> *Capture proposal:* two block pages side by side: `hpc-slurm-sqd-fulqrum` and the matching Fugaku or Miyabi HPC Profile block. They differ only in the HPC target, queue and executable map. If no second system is available for the capture, this image can be dropped and the text kept.

## Next steps

- [qcsc-prefect setup guide](TODO-qcsc-prefect-setup-guide) **(TODO: link)**
- [Sample-based quantum diagonalization of a chemistry Hamiltonian](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization)
- [Run quantum workloads with QRMI](https://quantum.cloud.ibm.com/docs/en/tutorials/run-quantum-workloads-with-qrmi)
- [qiskit-addon-sqd documentation](https://qiskit.github.io/qiskit-addon-sqd/)
- [QRMI](https://github.com/qiskit-community/qrmi) and
  [fulqrum](https://github.com/qiskit-community/fulqrum)
- [QRMI/Slurm sampler](https://qiskit-community.github.io/qcsc-prefect/howto/howto_qrmi_slurm_sampler/) and
  [Fulqrum integration](https://qiskit-community.github.io/qcsc-prefect/howto/howto_use_fulqrum_prefect/)
- [The SQD workflow source](https://github.com/qiskit-community/qcsc-prefect/tree/main/algorithms/open_loop/sqd)
- J. Robledo-Moreno *et al.*, "Chemistry beyond the scale of exact diagonalization on a
  quantum-centric supercomputer",
  [arXiv:2405.05068](https://arxiv.org/abs/2405.05068)